# Build a model to predict Koc 

The aim of this assignment is to build a model that can predict the soil sorption coefficient logKoc. In contrast to the previous assignment, this time the goal is outperform the state-of-the-art OPERA Koc model:

 - Paper on OPERA models, including the one for Koc: https://doi.org/10.1186/s13321-018-0263-1
 - Performance of the OPERA Koc model on the test set: R2 = 0.71, RMSE = 0.61
 - Model: weighted k-Nearest Neighbors regressor trained on 12 selected PaDEL descriptors (similar to RDKit descriptors)

As you are now a modelling expert, it is open to you which model architecture you use. BUT we ask you to justify your choices! 


#### Tasks:

1) Load the training data and split it into train and test according to the 'Tr_1_Tst_0' column

2) Think about a suitable architecture:
    - Suitable descriptors/fingerprints
    - Model choice (fine-tune a pretrained neural network? GNN? Ensemble method? Gaussian Process? ...?)
    - Consider providing an AD for your model (using an AD metric, or by defining the content of the training data (e.g., organic chemicals with a MW between x and y)
    - Consider providing prediction uncertainty: If you decide to do so, provide uncertainty calibration. If not, explain why.
    
        
3) Train the model on the same training data used in the paper (Tr_1_Tst_0 == 1). 

4) Consider tuning hyperparameters (e.g., using GridSearchCV and on a short list of parameters) 

5) Evaluate model on the test set (Tr_1_Tst_0 == 0), ONLY ONCE ! 

6) Compare your model performance on the test set to the OPERA model. 

#### Questions:
1) Which architecture did you choose, and why?
2) How well does your model perform on the test set? Could you outperform OPERA?
3) Did you add prediction uncertainty, and why? Are the uncertainties well calibrated?
4) Do you provide an AD, and why? How did you define your AD


1. GPR with RDKit descriptors. RDKit descriptors are similar to what OPERA uses (PaDEL), so I expected them to work well. feature selection via mutual information (top 30) + removal of highly correlated descriptors (>0.95) reduces noise and keeps GPR tractable. GPR was chosen mainly because it gives prediction uncertainty out of the box, no need to approximate it. kernel was selected via 5-fold GridSearchCV across RBF, Matern(nu=1.5) and Matern(nu=2.5) with WhiteKernel (AI suggested).
2. R2 = 0.805, RMSE = 0.505 vs OPERA R2 = 0.71, RMSE = 0.61. My setup outperforms the reference substiantially in both metrics.
3. yes, GPR gives uncertainty (std) directly as part of the prediction. calibration was done on a held-out 20% split of the training data: the calibration factor s = 1.06, meaning GPR is slightly overconfident. coverage improved from 0.917 to 0.936 after calibration, which is still a bit below the nominal 0.95. so calibration helps but isn't perfect, maybe due to the rather small calibration set.
4. yes, the AD is defined using the calibrated prediction uncertainty. molecules where y_std_calibrated > 0.507 (95th percentile of training uncertainties) are considered outside the AD. this is a handy choice for GPR since uncertainty already reflects how far a molecule is from the training data. 104/184 test molecules fall inside the AD (which are relatively few), with R2 = 0.838 and RMSE = 0.409. the fit is better than outside AD, with R2 = 0.777 and RMSE = 0.607, but not by a large factor. so the defined AD is somewhat meaningful, could however, probably be imporved.

#### 0. Import Packages

In [1]:
# import
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors

from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.base import clone

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF,
    Matern,
    WhiteKernel,
    ConstantKernel
)

from sklearn.metrics import r2_score, mean_squared_error

#### 1. Load training data

In [2]:
# Loading the data from the .sdf file
supplier = Chem.SDMolSupplier("KOC_QR.sdf")
df = pd.DataFrame([
    {
        "SMILES": Chem.MolToSmiles(m),
        **{p: m.GetProp(p) for p in ['preferred_name', 'LogKOC', 'Tr_1_Tst_0']}
    }
    for m in supplier if m is not None
])

In [3]:
# The data is pre-split in training and testing:
df.groupby('Tr_1_Tst_0').count()

,SMILES,preferred_name,LogKOC
Tr_1_Tst_0,,,
0,184,184,184
1,544,544,544


In [4]:
# Convert target to float
df["LogKOC"] = df["LogKOC"].astype(float)

# split into training and testing sets
train_df = df[df['Tr_1_Tst_0'] == '1'].copy()
test_df = df[df['Tr_1_Tst_0'] == '0'].copy()


print(f"Training set size: {len(train_df)}")
print(f"Testing set size: {len(test_df)}")

Training set size: 544
Testing set size: 184


#### 2. Build a supervised model of your choice

2.1 build feature matrix with RDKit descriptors. select the top 30 features with higher Mutual Information Score, then remove highly correlated features

In [5]:
# Extract all RDKit descriptors

descriptor_names = [desc[0] for desc in Descriptors._descList]

def calc_rdkit_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [np.nan] * len(descriptor_names)
    values = []
    for name, func in Descriptors._descList:
        try:
            values.append(func(mol))
        except:
            values.append(np.nan)
    return values

# Calculate descriptors
X_train_df = pd.DataFrame(
    [calc_rdkit_descriptors(s) for s in train_df["SMILES"]],
    columns=descriptor_names
)

X_test_df = pd.DataFrame(
    [calc_rdkit_descriptors(s) for s in test_df["SMILES"]],
    columns=descriptor_names
)

In [6]:
# filter descriptors 
print(f"Initial descriptor count: {X_train_df.shape[1]}")

# Remove descriptors containing NaNs
valid_cols = X_train_df.columns[~X_train_df.isna().any()]

X_train_df = X_train_df[valid_cols]
X_test_df  = X_test_df[valid_cols]

# Remove constant descriptors
nunique = X_train_df.nunique()
valid_cols = nunique[nunique > 1].index

X_train_df = X_train_df[valid_cols]
X_test_df  = X_test_df[valid_cols]

print(f"Descriptor count after cleaning: {X_train_df.shape[1]}")

Initial descriptor count: 217
Descriptor count after cleaning: 187


In [7]:
# Mutual Information feature selection
y_train = train_df["LogKOC"].values
y_test  = test_df["LogKOC"].values

mi_scores = mutual_info_regression(
    X_train_df,
    y_train,
    random_state=42
)

mi_df = pd.DataFrame({
    "descriptor": X_train_df.columns,
    "mi_score": mi_scores
})

mi_df = mi_df.sort_values(
    "mi_score",
    ascending=False
)

# Select top 30 descriptors
top_features = mi_df.head(30)["descriptor"].tolist()

print("Top selected descriptors:")
print(top_features)

X_train = X_train_df[top_features]
X_test  = X_test_df[top_features]

Top selected descriptors:
['MolLogP', 'Chi4v', 'Chi4n', 'Chi3v', 'PEOE_VSA6', 'SMR_VSA7', 'HeavyAtomMolWt', 'BertzCT', 'MolMR', 'MaxEStateIndex', 'MaxAbsEStateIndex', 'LabuteASA', 'ExactMolWt', 'NumValenceElectrons', 'MolWt', 'Chi2v', 'FpDensityMorgan1', 'Chi0', 'Chi1v', 'Chi3n', 'HeavyAtomCount', 'Chi0n', 'NumAromaticCarbocycles', 'fr_benzene', 'AvgIpc', 'Chi2n', 'Chi1', 'Chi1n', 'VSA_EState3', 'SlogP_VSA6']


In [8]:
# remove highly correlated descriptors
corr_matrix = X_train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# drop columns with correlation > 0.95, keeping only the one with the highest MI score
to_drop = set()
for column in upper.columns:
    high_corr = upper[column][upper[column] > 0.95].index.tolist()
    if high_corr:
        to_drop.update(high_corr)

X_train = X_train.drop(columns=to_drop)
X_test  = X_test.drop(columns=to_drop)

print("Removed correlated descriptors:")
print(to_drop)

print("\nFinal feature count:")
print(X_train.shape[1])

Removed correlated descriptors:
{'Chi4v', 'MolMR', 'LabuteASA', 'HeavyAtomCount', 'Chi0', 'Chi0n', 'Chi2n', 'Chi1', 'ExactMolWt', 'Chi3n', 'NumValenceElectrons', 'NumAromaticCarbocycles', 'HeavyAtomMolWt', 'MaxEStateIndex', 'Chi4n'}

Final feature count:
15


2.2 Gaussian Process Regression: Define pipeline, param_grid, GridSearchCV, and find best_model

In [9]:
# Define GPR pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("gpr", GaussianProcessRegressor(
        random_state=42,
        normalize_y=True
    ))
])

In [10]:
# Define parameter grid
param_grid = [
    {
        "gpr__kernel": [
            ConstantKernel(1.0) * RBF(length_scale=1.0) + WhiteKernel()
        ],
        "gpr__alpha": [1e-6, 1e-4, 1e-2],
        "gpr__n_restarts_optimizer": [5, 10]
    },
    {
        "gpr__kernel": [
            ConstantKernel(1.0) * Matern(length_scale=1.0, nu=1.5) + WhiteKernel()
        ],
        "gpr__alpha": [1e-6, 1e-4, 1e-2],
        "gpr__n_restarts_optimizer": [5, 10]
    },
    {
        "gpr__kernel": [
            ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5) + WhiteKernel()
        ],
        "gpr__alpha": [1e-6, 1e-4, 1e-2],
        "gpr__n_restarts_optimizer": [5, 10]
    }
]

In [11]:
# Grid Search
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

Fitting 5 folds for each of 18 candidates, totalling 90 fits


[CV] END gpr__alpha=1e-06, gpr__kernel=1**2 * RBF(length_scale=1) + WhiteKernel(noise_level=1), gpr__n_restarts_optimizer=5; total time=  13.4s
[CV] END gpr__alpha=1e-06, gpr__kernel=1**2 * RBF(length_scale=1) + WhiteKernel(noise_level=1), gpr__n_restarts_optimizer=5; total time=  13.4s
[CV] END gpr__alpha=1e-06, gpr__kernel=1**2 * RBF(length_scale=1) + WhiteKernel(noise_level=1), gpr__n_restarts_optimizer=5; total time=  13.5s
[CV] END gpr__alpha=1e-06, gpr__kernel=1**2 * RBF(length_scale=1) + WhiteKernel(noise_level=1), gpr__n_restarts_optimizer=5; total time=  13.7s
[CV] END gpr__alpha=1e-06, gpr__kernel=1**2 * RBF(length_scale=1) + WhiteKernel(noise_level=1), gpr__n_restarts_optimizer=5; total time=  14.0s
[CV] END gpr__alpha=0.0001, gpr__kernel=1**2 * RBF(length_scale=1) + WhiteKernel(noise_level=1), gpr__n_restarts_optimizer=5; total time=  13.3s
[CV] END gpr__alpha=0.0001, gpr__kernel=1**2 * RBF(length_scale=1) + WhiteKernel(noise_level=1), gpr__n_restarts_optimizer=5; total tim

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'gpr__alpha': [1e-06, 0.0001, ...], 'gpr__kernel': [1**2 * RBF(le...noise_level=1)], 'gpr__n_restarts_optimizer': [5, 10]}, {'gpr__alpha': [1e-06, 0.0001, ...], 'gpr__kernel': [1**2 * Matern...noise_level=1)], 'gpr__n_restarts_optimizer': [5, 10]}, ...]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed 

In [12]:
# Best model
best_model = grid.best_estimator_

print("\nBest parameters:")
print(grid.best_params_)

print("\nBest CV RMSE:")
print(-grid.best_score_)


Best parameters:
{'gpr__alpha': 0.01, 'gpr__kernel': 1**2 * Matern(length_scale=1, nu=1.5) + WhiteKernel(noise_level=1), 'gpr__n_restarts_optimizer': 10}

Best CV RMSE:
0.6287395203121995


2.3 uncertainty calibration

In [13]:
# held-out calibration split
X_fit, X_cal, y_fit, y_cal = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# single GPR fit with best params on X_fit
cal_model = clone(pipeline)
cal_model.set_params(**grid.best_params_)
cal_model.fit(X_fit, y_fit)

# calibration factor
y_pred_cal, y_std_cal = cal_model.predict(X_cal, return_std=True)
s = np.std((y_cal - y_pred_cal) / y_std_cal)

print(f"Calibration factor s = {s:.3f}")
print("(s > 1 → GPR overconfident, s < 1 → underconfident)")

# coverage check
def coverage(y_true, y_pred, y_std, factor=1.0):
    lower = y_pred - 1.96 * factor * y_std
    upper = y_pred + 1.96 * factor * y_std
    return np.mean((y_true >= lower) & (y_true <= upper))

print(f"Coverage before: {coverage(y_cal, y_pred_cal, y_std_cal):.3f}  (nominal 0.95)")
print(f"Coverage after:  {coverage(y_cal, y_pred_cal, y_std_cal, s):.3f}  (nominal 0.95)")

Calibration factor s = 1.060
(s > 1 → GPR overconfident, s < 1 → underconfident)
Coverage before: 0.917  (nominal 0.95)
Coverage after:  0.936  (nominal 0.95)


In [14]:
# AD threshold: 95th percentile of calibrated training uncertainty
_, y_std_train = best_model.predict(X_train, return_std=True)
y_std_train_cal = s * y_std_train
ad_threshold = np.percentile(y_std_train_cal, 95)
print(f"AD threshold (95th percentile): {ad_threshold:.3f}")

AD threshold (95th percentile): 0.507


#### 3. Model performance on test set
--> **TEST ONLY ONCE !**

In [15]:
# Predict
y_pred, y_std = best_model.predict(X_test, return_std=True)
y_std_calibrated = s * y_std

# Metrics
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R2   : {r2:.3f}")
print(f"RMSE : {rmse:.3f}")

# Prediction intervals
lower = y_pred - 1.96 * y_std_calibrated
upper = y_pred + 1.96 * y_std_calibrated

R2   : 0.805
RMSE : 0.505


In [16]:
results_df = pd.DataFrame({
    "Experimental"   : y_test,
    "Predicted"      : y_pred,
    "Uncertainty_STD": y_std_calibrated,
    "Lower_95CI"     : lower,
    "Upper_95CI"     : upper
})

results_df.head()

,Experimental,Predicted,Uncertainty_STD,Lower_95CI,Upper_95CI
0,1.40,1.775766,0.424678,0.943398,2.608134
1,2.01,2.667769,0.651901,1.390042,3.945496
2,1.85,1.773434,0.603804,0.589978,2.956891
3,5.30,5.365823,0.495006,4.395611,6.336036
4,5.37,5.795049,0.543813,4.729176,6.860922


In [17]:
# Applicability Domain
inside_ad = y_std_calibrated <= ad_threshold

print(f"Molecules inside AD:  {inside_ad.sum()} / {len(inside_ad)}")
print(f"Performance inside AD:")
print(f"  R2   : {r2_score(y_test[inside_ad], y_pred[inside_ad]):.3f}")
print(f"  RMSE : {np.sqrt(mean_squared_error(y_test[inside_ad], y_pred[inside_ad])):.3f}")

outside_ad = ~inside_ad

print(f"Molecules outside AD: {outside_ad.sum()} / {len(outside_ad)}")
print(f"Performance outside AD:")
print(f"  R2   : {r2_score(y_test[outside_ad], y_pred[outside_ad]):.3f}")
print(f"  RMSE : {np.sqrt(mean_squared_error(y_test[outside_ad], y_pred[outside_ad])):.3f}")

Molecules inside AD:  104 / 184
Performance inside AD:
  R2   : 0.838
  RMSE : 0.409
Molecules outside AD: 80 / 184
Performance outside AD:
  R2   : 0.777
  RMSE : 0.607
